[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/quadrics/gaussian/gaussian.ipynb)

# A Gaussian from Point Moments

This example fits a Gaussian distribution to an uncentred point cloud and draws its one-sigma contour directly from homogeneous point moments. Each sample point is modelled with unit weight, so averaging the outer product of every point with itself records both the cloud's location and its directional spread in a single moment map without subtracting a sample mean or solving for principal axes first. Inverting that moment map yields a precision map that measures how many standard deviations any query point lies from the cloud.

In projective geometric algebra, each unit-weight point contributes the rank-one dyad `points * (Plane & points)` mapping planes to points, and the cloud mean of those dyads inverts directly to the `precision` polarity from points to planes. Pairing a unit-weight sample with its image `precision(samples) & samples` evaluates to one plus the squared Mahalanobis distance, whose Gaussian exponential peaks at one at the cloud mean. Applying `precision` to the mean point recovers the unit-weight plane `weight`, and `precision - 2 * weight * (weight & Point)` shifts the zero locus of the quadratic form to the one-sigma quadric.

In matrix notation, fitting a multivariate normal computes the sample mean $\boldsymbol\mu = \frac{1}{N}\sum_{i=1}^N \mathbf x_i$ and the centred covariance $\Sigma = \frac{1}{N}\sum_{i=1}^N (\mathbf x_i - \boldsymbol\mu)(\mathbf x_i - \boldsymbol\mu)^{\mathsf T}$ separately to evaluate $(\mathbf x - \boldsymbol\mu)^{\mathsf T}\Sigma^{-1}(\mathbf x - \boldsymbol\mu)$. Packing homogeneous coordinates $\tilde{\mathbf x} = (\mathbf x^{\mathsf T}, 1)^{\mathsf T}$ collects those pieces into the block moment matrix $M = \begin{pmatrix}\Sigma + \boldsymbol\mu\boldsymbol\mu^{\mathsf T} & \boldsymbol\mu \\ \boldsymbol\mu^{\mathsf T} & 1\end{pmatrix}$ and the one-sigma quadric matrix $Q = \begin{pmatrix}\Sigma^{-1} & -\Sigma^{-1}\boldsymbol\mu \\ -\boldsymbol\mu^{\mathsf T}\Sigma^{-1} & \boldsymbol\mu^{\mathsf T}\Sigma^{-1}\boldsymbol\mu - 1\end{pmatrix}$ satisfying $\tilde{\mathbf x}^{\mathsf T} Q \tilde{\mathbf x} = 0$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
%matplotlib inline
from functools import partial

import numpy as np

from numga import NumpyContext
from numga.algebras import PGA2D as ga

from examples.quadrics.gaussian import render
from examples.util.animation import save_figure

mv = NumpyContext(ga).multivector
Point = ga.gatype.antivector()                      # Antivector-
Plane = ga.gatype.vector()                          # Vector+
Scalar = ga.gatype.scalar()                         # Scalar+
Polarity = ga.gatype((Plane, Point))                # Plane+ <- Point-

## Moment precision and one-sigma quadric

Placing an anisotropic sample cloud by a translation and a rotation moves its centre off the origin and tilts its principal axes away from the coordinate directions. Because motors act linearly on homogeneous points, transforming the cloud transforms its second moment, precision polarity, and one-sigma quadric by the same motor without reference to the coordinate origin.

In [ ]:
n_points, spread, seed = 400, np.array([1.5, 0.5]), 4
shift, turn = (0.6, 0.3), 0.3

rest_points = mv.point(np.random.default_rng(seed).normal(size=(n_points, 2)) * spread)  # [n_points] Point-
placement = (mv.xw * shift[0] - mv.yw * shift[1]).exp() * (mv.xy * turn).exp()           # [] Motor+
points = placement >> rest_points                                                        # [n_points] Point-

In [ ]:
def gaussian_fit(points: Point) -> tuple[Polarity, Polarity]:
    """Homogeneous point moments determine the precision polarity (`[] Plane+ <- Point-`) and one-sigma quadric (`[] Plane+ <- Point-`) of a point cloud."""
    # Each point contributes its rank-one dyad from planes to points; their mean holds both location and spread without centring.
    moment = (points * (Plane & points)).mean(axis=0)                          # [] Point- <- Plane+
    precision = moment.inverse()                                               # [] Plane+ <- Point-
    # Pairing the recovered weight plane with any unit-weight point gives 1, so subtracting its dyad twice puts the zero locus at Mahalanobis distance 1.
    weight = precision(points.mean(axis=0))                                    # [] Plane+
    return precision, precision - 2 * weight * (weight & Point)


def gaussian_density(precision: Polarity, samples: Point) -> Scalar:
    """A precision polarity evaluates on unit-weight sample points (`[...] Point-`) to the peak-normalized Gaussian density (`[...] Scalar+`)."""
    squared_distance = (precision(samples) & samples) - 1                      # [...] Scalar+
    return (-0.5 * squared_distance).exp()


precision, ellipsoid = gaussian_fit(points)                                    # [] Plane+ <- Point-, Plane+ <- Point-
save_figure(render.gaussian_figure(points, partial(gaussian_density, precision), ellipsoid), "gaussian");

Blue shading shows the fitted Gaussian density peaking at the cloud's mean, dark markers show the displaced and rotated sample points, and the orange curve traces the zero contour of the one-sigma quadric around the core of the cloud away from the faint coordinate axes through the origin.

In [ ]:
# checks
mean = points.mean(axis=0)                                                     # [] Point-
levels = ellipsoid(points) & points                                            # [n_points] Scalar+
assert (gaussian_density(precision, mean) - 1).abs() < 1e-8
assert ((ellipsoid(mean) & mean) + 1).abs() < 1e-8
assert (levels.mean(axis=0) - 1).abs() < 1e-8
assert ((levels + 2 * gaussian_density(precision, points).log() + 1).abs() < 1e-8).all()